# ESM-IF1 Inverse Folding

This notebook demonstrates both functions of the ESM-IF1 tool. In the first section, we use `run_esm_if1_sample` to generate new amino acid sequences conditioned on a target backbone structure. In the second section, we use `run_esm_if1_score` to evaluate how well a given sequence fits a target structure by computing average log-likelihood and perplexity.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm_if1")
display_overview("esm_if1")
display_docs_section("esm_if1", "Background")

# ESM-IF1 / ProteinDPO

> ESM-IF1 is a language-model-based inverse folding model that designs protein sequences conditioned on backbone structure. It uses a GVP-GNN encoder to represent 3D structure and a Transformer decoder to autoregressively generate sequences. ProteinDPO is a fine-tuned variant aligned to experimental fitness data via Direct Preference Optimization, optimized for designing stable proteins.

## Background

[Inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding) solves the "reverse" protein design problem: given a desired 3D backbone structure, what amino acid sequence will fold into that structure?

ESM-IF1 (Hsu et al., 2022) takes a different architectural approach from message-passing models like ProteinMPNN. It uses:

1. **GVP-GNN encoder**: A [Geometric Vector Perceptron](https://arxiv.org/abs/2009.01411) graph neural network encodes the backbone structure into per-residue representations that capture both scalar and geometric features.
2. **Transformer decoder**: A standard autoregressive Transformer generates the amino acid sequence one token at a time, attending to the structural encoding. This leverages the same architecture behind large language models.
3. **Training on predicted structures**: Unlike ProteinMPNN (trained on \~19K experimental structures), ESM-IF1 was trained on 12M predicted structures from ESM, giving it broader coverage of protein fold space.

[ProteinDPO](https://doi.org/10.1101/2024.05.20.595026) (Widatalla, Rafailov & Hie, 2024) fine-tunes ESM-IF1 using [Direct Preference Optimization](https://arxiv.org/abs/2305.18290) on experimental fitness data, aligning the model to prefer sequences with higher measured stability. This is analogous to RLHF in language models -- the model learns to generate sequences that score well on real experimental assays.

## Available tools

In [2]:
display_available_tools("esm_if1")

- **`run_esm_if1_sample()`** — Sample protein sequences conditioned on backbone structure using ESM-IF1 or ProteinDPO (DPO-aligned for stability). Supports multi-chain complexes.
- **`run_esm_if1_score()`** — Score protein sequences against backbone structures using ESM-IF1 or ProteinDPO. Computes average log-likelihood and perplexity with multi-chain complex support.

### `run_esm_if1_sample`

Given a backbone structure, ESM-IF1 generates new amino acid sequences that are predicted to fold into the target conformation. You can choose between vanilla ESM-IF1 weights or the ProteinDPO variant, which is optimized for designing thermodynamically stable proteins. Sampling temperature controls the diversity of the generated sequences, with lower temperatures producing more conservative designs that closely match the structural constraints.

In [3]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1SampleConfig,
    ESMIF1SampleInput,
    run_esm_if1_sample,
)
from proto_tools.tools.inverse_folding.shared_data_models import (
    InverseFoldingInput,
    InverseFoldingStructureInput,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chains_to_redesign=["A"],
)
inputs = InverseFoldingInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[InverseFoldingStructureInput]` | required | List of InverseFoldingInput objects, each containing a structure and optional chain\_ids/fixed\_positions constraints. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. Automatically converted to Structure. |
| `chain_ids` | `array` |  | Optional list of chain IDs to design. If None, all chains in the structure are designed. |
| `fixed_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions to keep fixed during design. Positions are 1-indexed. |

In [5]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_sample")

# Configure sampling | see docs above for all fields
config = ESMIF1SampleConfig(
    num_sequences_per_structure=3,
    temperature=0.1,
    weights_variant="protein_dpo",  # or "esmif" for vanilla ESM-IF1
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESMIF1SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `weights_variant` | `enum` | `protein_dpo` | Which model weights to use. 'esmif' loads vanilla ESM-IF1, 'protein\_dpo' loads DPO-aligned weights optimized for protein stability. Available options: `esmif`, `protein_dpo` |
| `seed` | `integer` |  | Random seed for sampling reproducibility. |
| `num_sequences_per_structure` | `integer` | `1` | Total number of sequences to generate per structure. |
| `batch_size` | `integer` |  | Number of sequences to process simultaneously on GPU. |
| `temperature` | `number` | `0.1` | Controls randomness in sampling from logits. |
| `excluded_amino_acids` | `array` |  | Amino acids disallowed in the designed sequence. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution), or specific GPU devices like 'cuda:0'. Defaults to 'cuda'. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
result = run_esm_if1_sample(inputs, config)

ESM-IF1 sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_sample")

# Print designed sequences and their log-likelihoods
for i, seq_res in enumerate(result.designed_sequences):
    for j, seq in enumerate(seq_res.sequences):
        ll = seq_res.log_likelihoods[j]
        display_seq = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Log-likelihood: {ll:.4f}")

**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `List[DesignedSequences]` | required | List of DesignedSequences objects, one for each input structure. The order matches the input structures order. |
| `sequences` | `List[string]` | required | Designed amino acid sequences in single-letter code. |

Structure 0, Design 1: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3543
Structure 0, Design 2: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3543
Structure 0, Design 3: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -0.3592


### `run_esm_if1_score`

ESM-IF1 can evaluate how well a given amino acid sequence fits a target backbone structure by computing average log-likelihood and perplexity. This is useful for ranking candidate sequences from other design methods, comparing wild-type to mutant sequences, or validating that a designed sequence is structurally compatible with the intended fold. Scoring uses the full complex context, so multi-chain interactions are accounted for.

In [8]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1ScoringConfig,
    ESMIF1ScoringInput,
    run_esm_if1_score,
)
from proto_tools.tools.inverse_folding.shared_data_models import SequenceStructurePair

In [9]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_score")

# Score the first designed sequence against the GFP backbone
designed_seq = result.designed_sequences[0].sequences[0]

scoring_input = ESMIF1ScoringInput(
    sequence_structure_pairs=[
        SequenceStructurePair(
            sequence=designed_seq,
            structure=gfp_structure,
        )
    ]
)

**Input** — `ESMIF1ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence_structure_pairs` | `List[SequenceStructurePair]` | required | List of sequence-structure pairs to score. Each pair contains a sequence and a structure to score the sequence against. |
| `sequence` | `string` | required | Protein sequence to score against the structure. |
| `structure` | `Structure` | required | Protein structure to score the sequence against. |

In [10]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_score")

# Configure scoring | see docs above for all fields
scoring_config = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESMIF1ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `weights_variant` | `enum` | `protein_dpo` | Which model weights to use. 'esmif' loads vanilla ESM-IF1, 'protein\_dpo' loads DPO-aligned weights optimized for protein stability. Available options: `esmif`, `protein_dpo` |
| `seed` | `integer` |  | Random seed. When set, tools run reproducibly up to small GPU float noise (see `BaseToolOutput.approx_equal`). When None, tools take their fast non-deterministic path. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the scoring function
score_result = run_esm_if1_score(scoring_input, scoring_config)

ESM-IF1 scoring:   0%|          | 0/1 [00:00<?, ?pair/s]

In [12]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_score")

# Print scores for the designed sequence
print(f"Avg log-likelihood: {score_result.scores[0].avg_log_likelihood:.4f}")
print(f"Perplexity: {score_result.scores[0].perplexity:.4f}")

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[InverseFoldingScoringMetrics]` | required | List of scoring outputs, one per input sequence-structure pair. Each entry is a `Metrics` subclass with scalar metrics (accessed via `score.perplexity` or `score["perplexity"]`) plus declared `logits` / `vocab` fields. |
| `logits` | `array` |  | Per-position logits array `(seq_len, vocab_size)`. `None` unless the tool returns logits. |
| `vocab` | `array` |  | Token ordering for `logits`. |
| `primary_metric` | `string` |  | Name of the metric that best summarizes the result overall (e.g. `"avg_plddt"` for AlphaFold2). Used by downstream UI and reporting to pick a headline value. |

Avg log-likelihood: -0.3543
Perplexity: 1.4252


## Export Results

Sampling results can be exported to FASTA format for use in downstream sequence analysis pipelines. Scoring results can be exported to CSV or JSON format for further analysis or comparison.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="esm_if1_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'esm_if1_sequences.fasta'}")

score_result.export(name="esm_if1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'esm_if1_scores.csv'}")

Designed sequences exported to example_output/esm_if1_sequences.fasta
Scores exported to example_output/esm_if1_scores.csv
